In [38]:
import pandas as pd
import numpy as np

In [39]:
df = pd.read_csv('OnlineRetail.csv', encoding='latin1')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [40]:
df.info() #hasil tipe data

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [41]:
#Mengecek apakah ada missing value (NaN) di setiap kolom.
#Mengecek apakah ada duplikat baris di dataset.

print("Missing Value per Column: \n", df.isnull().sum())
print("Jumlah Duplikat: ", df.duplicated().sum())

Missing Value per Column: 
 InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64
Jumlah Duplikat:  5268


In [42]:
print(df.nunique()) #jumlah nilai masing masing kolom

InvoiceNo      25900
StockCode       4070
Description     4223
Quantity         722
InvoiceDate    23260
UnitPrice       1630
CustomerID      4372
Country           38
dtype: int64


membuat summary data quality berupa:

- Nama kolom (column)
- Tipe data (types)
- Jumlah nilai unik (distinct)
- Jumlah missing value (count_na)
- Persentase missing value (percent_na)

In [43]:
types = df.dtypes
distinct = df.nunique()
count_na = df.isna().sum()
percent_na = round((df.isna().sum()/len(df))*100, 3)

print('dimensi data: ', df.shape)

df_check = pd.concat([types, distinct, count_na, percent_na], axis=1)
df_check.reset_index(level=0, inplace=True)
df_check.rename(columns = {'index':'column', 0:'types', 1:'distinct', 2:'count_na', 3: 'percent_na'}, inplace = True)
df_check

dimensi data:  (541909, 8)


,column,types,distinct,count_na,percent_na
0,InvoiceNo,object,25900,0,0.000
1,StockCode,object,4070,0,0.000
2,Description,object,4223,1454,0.268
3,Quantity,int64,722,0,0.000
4,InvoiceDate,object,23260,0,0.000
5,UnitPrice,float64,1630,0,0.000
6,CustomerID,float64,4372,135080,24.927
7,Country,object,38,0,0.000


Fungsi ini:

Memberikan ringkasan cepat kualitas data.

Bisa dipanggil dengan check_missing_summary(df) → langsung dapat tabel berisi info tipe data, jumlah nilai unik, missing values, dan persentasenya.

Berguna untuk data cleaning tahap awal sebelum masuk ke preprocessing / modelling.

In [44]:
# buat fungsi untuk cek all in once

def check_missing_summary(df):
    """
    Mengembalikan ringkasan informasi dataframe:
    - Tipe data
    - Jumlah nilai unik (distinct)
    - Jumlah missing value
    - Persentase missing value
    """
    types = df.dtypes
    distinct = df.nunique()
    count_na = df.isna().sum()
    percent_na = round((df.isna().sum() / len(df)) * 100, 3)

    print('Dimensi data:', df.shape)

    df_check = pd.concat([types, distinct, count_na, percent_na], axis=1)
    df_check.reset_index(level=0, inplace=True)
    df_check.columns = ['column', 'types', 'distinct', 'count_na', 'percent_na']

    return df_check

In [45]:
# Hitung total baris yang memiliki missing value
num_rows_with_na = df.isna().any(axis=1).sum()
print(f"Jumlah baris dengan missing value: {num_rows_with_na}")

Jumlah baris dengan missing value: 135080


Menampilkan:

- Berapa baris dengan hanya Description kosong.

- Berapa baris dengan hanya CustomerID kosong.

- Berapa baris dengan keduanya kosong.

- Total baris yang akan dihapus.

- Persentase baris yang dihapus terhadap total data.

In [46]:
# Hanya desctiption yang NaN
only_description = df[df["Description"].isna() & df["CustomerID"].notna()]

# Hanya CustomerID yang NaN
only_customerID = df[df["CustomerID"].isna() & df["Description"].notna()]

# Keduanya NaN
both_missing = df[df["Description"].isna() & df["CustomerID"].isna()]

# Gabungkan total baris yang akan dihapus
rows_to_drop = df[df["Description"].isna() | df["CustomerID"].isna()]

print("🔍 Ringkasan kombinasi missing:")
print(f"Hanya Description    : {only_description.shape[0]} baris")
print(f"Hanya accident     : {only_customerID.shape[0]} baris")
print(f"Keduanya missing   : {both_missing.shape[0]} baris")
print(f"Total rows to drop : {rows_to_drop.shape[0]} dari {df.shape[0]} baris")
print(f"Persentase drop    : {100 * rows_to_drop.shape[0] / df.shape[0]:.2f}%")

🔍 Ringkasan kombinasi missing:
Hanya Description    : 0 baris
Hanya accident     : 133626 baris
Keduanya missing   : 1454 baris
Total rows to drop : 135080 dari 541909 baris
Persentase drop    : 24.93%


In [47]:
# Buat copy df dan drop baris yang bermasalah tadi
df_cleaned = df.drop(rows_to_drop.index)

In [48]:
# Cek hasil
print(f"Jumlah baris awal: {len(df)}")
print(f"Jumlah baris setelah drop: {len(df_cleaned)}")

check_missing_summary(df_cleaned)

Jumlah baris awal: 541909
Jumlah baris setelah drop: 406829
Dimensi data: (406829, 8)


,column,types,distinct,count_na,percent_na
0,InvoiceNo,object,22190,0,0.0
1,StockCode,object,3684,0,0.0
2,Description,object,3896,0,0.0
3,Quantity,int64,436,0,0.0
4,InvoiceDate,object,20460,0,0.0
5,UnitPrice,float64,620,0,0.0
6,CustomerID,float64,4372,0,0.0
7,Country,object,37,0,0.0


In [49]:
df_cleaned.describe(include='all')

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
count,406829,406829,406829,406829.000000,406829,406829.000000,406829.000000,406829
unique,22190,3684,3896,NaN,20460,NaN,NaN,37
top,576339,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,11/14/2011 15:27,NaN,NaN,United Kingdom
freq,542,2077,2070,NaN,543,NaN,NaN,361878
mean,NaN,NaN,NaN,12.061303,NaN,3.460471,15287.690570,NaN
std,NaN,NaN,NaN,248.693370,NaN,69.315162,1713.600303,NaN
min,NaN,NaN,NaN,-80995.000000,NaN,0.000000,12346.000000,NaN
25%,NaN,NaN,NaN,2.000000,NaN,1.250000,13953.000000,NaN
50%,NaN,NaN,NaN,5.000000,NaN,1.950000,15152.000000,NaN
75%,NaN,NaN,NaN,12.000000,NaN,3.750000,16791.000000,NaN


In [50]:
# isi missing Description dengan "Unknown"
df['Description'] = df['Description'].fillna("Unknown")
df.isnull().sum()

,0
InvoiceNo,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


In [51]:
# CustomerID drop baris yg NaN
df = df.dropna(subset=['CustomerID'])
df.isnull().sum()

,0
InvoiceNo,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,0
Country,0


In [52]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


membuat:

- TotalPrice → nilai transaksi per baris.

- InvoiceYear, InvoiceMonth, InvoiceDay, InvoiceHour → informasi waktu transaksi.

👉 Dengan tambahan fitur ini, dataset jadi lebih siap untuk analisis tren dan modelling.

In [53]:
# 7. FEATURE ENGINEERING
# =======================================================
# (a) Buat kolom TotalPrice = Quantity * UnitPrice
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# (b) Ubah InvoiceDate ke tipe datetime & extract fitur waktu
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['InvoiceYear'] = df['InvoiceDate'].dt.year
df['InvoiceMonth'] = df['InvoiceDate'].dt.month
df['InvoiceDay'] = df['InvoiceDate'].dt.day
df['InvoiceHour'] = df['InvoiceDate'].dt.hour

In [54]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,InvoiceYear,InvoiceMonth,InvoiceDay,InvoiceHour
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010,12,1,8
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,8
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010,12,1,8
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,8
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,8


In [56]:
### Save new dataset to excel

from google.colab import drive
drive.mount('/content/drive')


df.to_excel('Haris_project_wrangler_online_retail.xlsx', index=False)


!cp Haris_project_wrangler_online_retail.xlsx "/content/drive/My Drive/Colab Notebooks/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
